In [1]:
from bs4 import BeautifulSoup, NavigableString
import openai
import os
import re
import html


In [2]:
def replace_images_with_markdown(problem_soup):
    images = problem_soup.find_all('img')
    for idx, img in enumerate(images, start=1):
        img.replace_with(f"![image]({idx}.png)")
    return problem_soup

In [3]:
def rewrite_formula_and_lists(soup):
    for sub_tag in soup.find_all('sub'):
        sub_tag.replace_with('_{' + sub_tag.text + '}')

    for sup_tag in soup.find_all('sup'):
        sup_tag.replace_with('^{' + sup_tag.text + '}')

    for var_tag in soup.find_all('var'):
        var_tag.replace_with('$' + var_tag.text + '$')

    for li_tag in soup.find_all('li'):
        li_tag.replace_with('- ' + li_tag.text)
    return soup

In [4]:
import os
import json
import tqdm


def load_problems(problems_folder):
    """
    Scan the folder to retrieve problems with data.
    """
    subdirs = [os.path.join(problems_folder, d) for d in os.listdir(problems_folder) 
               if os.path.isdir(os.path.join(problems_folder, d))]
    return subdirs

def load_problem_data(folder_path):
    """
    Load the data.json file from the specified folder path.
    """
    with open(os.path.join(folder_path, 'data.json'), 'r') as file:
        data = json.load(file)
    return data

def save_new_data(folder_path, new_data):
    """
    Save the modified data to a new file in the folder path.
    """
    with open(os.path.join(folder_path, 'new_data.json'), 'w') as file:
        json.dump(new_data, file, indent=4)

def process_problem(folder_path, active_prefixes=None):
    prefix_to_function = {
        'ac': "process_atcoder",
        'az': "process_aizu",
        'cc': "process_codechef",
        'cf': "process_codeforces",
        'cw': "process_codewars",
        'ep': "process_euler",
        'g4g': "process_geeksforgeeks",
        'hr': "process_hackerrank",
        'lc': "process_leetcode",
        'ok': "process_openkattis"
    }

    # If no specific prefixes are specified, all are considered active
    if active_prefixes is None:
        active_prefixes = prefix_to_function.keys()

    prefix = os.path.basename(folder_path).split('_')[0]
    if prefix in prefix_to_function and prefix in active_prefixes:
        globals()[prefix_to_function[prefix]](folder_path)
    elif prefix not in active_prefixes:
        pass
    else:
        print(f"Unknown prefix: {prefix}")

In [32]:
def process_codeforces(folder_path):
    data = load_problem_data(folder_path)
    new_data = {} 
    new_data.update(data)
    # Question text
    if "raw_problem" not in new_data:
        # Read from original crawl
        crawled_data = load_problem_data(os.path.join("/home/kaixin/Desktop/mmcode/crawl/crawled/codeforces/problems", new_data["problem_id"]))
        new_data["raw_problem"] = crawled_data["problem_raw"]
    raw_problem = new_data["raw_problem"]
    soup = BeautifulSoup(raw_problem, 'html.parser')
    replace_images_with_markdown(soup)
    #   Remove header
    soup.find('div', class_='header').decompose()
    rewrite_formula_and_lists(soup)
    #   Insert a newline after each paragraph tag
    for p in soup.find_all(['p', 'div', 'pre']):
        p.append(soup.new_string('\n'))
    # Replace <br> with newline characters
    for br in soup.find_all("br"):
        br.replace_with("\n")
    
    new_data["question"] = soup.text
    
    
    save_new_data(folder_path, new_data)

# Run

In [30]:
problems = load_problems("/home/kaixin/Desktop/mmcode/mmcode_dataset")
active_prefixes = ['cf']

for problem in tqdm.tqdm(problems):
    process_problem(problem, active_prefixes=active_prefixes)

100%|██████████| 3548/3548 [00:08<00:00, 396.03it/s]


In [33]:
# Write a function to iterate all the cf problems under the folder and copy the new_data.json as data.json.
for subdir in os.listdir("/home/kaixin/Desktop/mmcode/mmcode_dataset"):
    if subdir.startswith("ep"):
        data = load_problem_data(os.path.join("/home/kaixin/Desktop/mmcode/mmcode_dataset", subdir))
        data["problem_id"] = f"pe{subdir[2:]}"
        with open(os.path.join("/home/kaixin/Desktop/mmcode/mmcode_dataset", subdir, "data.json"), 'w') as file:
            json.dump(data, file, indent=4)
        # rename the folder to start with pe
        os.rename(f"/home/kaixin/Desktop/mmcode/mmcode_dataset/{subdir}", f"/home/kaixin/Desktop/mmcode/mmcode_dataset/pe{subdir[2:]}")